In [1]:
Output = '/Users/alexis/Library/CloudStorage/OneDrive-UniversityofNorthCarolinaatChapelHill/CEMALB_DataAnalysisPM/Projects/P1016. THC Adducts/P1016.3. Analyses/P1016.3.4. Adductome Integration/Output'
cur_date = "111126"

library(readxl)
library(openxlsx)
library(tidyverse)
library(reshape2)
library(org.Hs.eg.db)

# reading in files
results_df = data.frame(read_excel("Input/DESeq2_Results_110726.xlsx"))
thc_adducts_df = data.frame(read_excel("Input/THC_Adducts_111026.xlsx"))
upstream_reg_df = data.frame(read_excel("Input/Upstream_Regulators_111126.xls"))

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.2     ✔ tibble    3.3.0
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.1.0     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attaching package: ‘reshape2’


The following object is masked from ‘package:tidyr’:

    smiths


Loading required package: AnnotationDbi

Loading required package: stats4

Loading required package: BiocGenerics

Loading required package: generics


Attaching package: ‘generics’


The following object is masked from ‘package:lubridate’:

    as.difftime


The following object is masked from ‘package:dplyr’:

    explain


The following objects are masked from ‘package:b

In [2]:
head(results_df)
head(thc_adducts_df)
head(upstream_reg_df)

,Gene,log2FC,P.Value,P.Adj,Comparison
,<chr>,<dbl>,<dbl>,<dbl>,<chr>
1,ENSG00000000003,-0.007988413,0.9461493,1,D8THC vs. Control
2,ENSG00000000419,0.075514481,0.6994330,1,D8THC vs. Control
3,ENSG00000000457,0.119709242,0.6461443,1,D8THC vs. Control
4,ENSG00000000460,0.125056396,0.7907686,1,D8THC vs. Control
5,ENSG00000000938,1.111036342,0.4973247,1,D8THC vs. Control
6,ENSG00000000971,0.173825486,0.8245174,1,D8THC vs. Control


,Gene_Name
,<chr>
1,ALB
2,ARAP1
3,ATL3
4,ATP6V0A4
5,CAAP1
6,CARD11


Upstream_Regulator Expr_Log_Ratio Molecule_Type                  
1 YAP1               -0.694         transcription regulator        
2 RICTOR             -0.147         other                          
3 torin1                 NA         chemical reagent               
4 SPEN               -1.004         transcription regulator        
5 beta-estradiol         NA         chemical - endogenous mammalian
6 HNF4A               0.634         transcription regulator        
  Predicted_Activation_State Activation.z.score p.value.of.overlap
1 NA                         -1.687             1.12e-42          
2 Inhibited                  -9.432             1.47e-40          
3 Inhibited                  -5.785             7.83e-39          
4 Activated                   6.530             5.95e-31          
5 Activated                   3.275             3.20e-28          
6 NA                          1.153             1.69e-27          
  Target_Molecules                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               

In [3]:
# creating a df that contains the number of times the genes were targeted by upstream regulators
cleaned_reg_df = upstream_reg_df[,c(1,3,4,7)] %>%
    dplyr::select(Target_Molecules) %>%
    separate_rows(Target_Molecules, sep = ",") %>%
    # removes whitespace from a string
    mutate(Target_Molecules = str_trim(Target_Molecules)) %>%
    count(Target_Molecules, sort = TRUE)
head(cleaned_reg_df)

Target_Molecules,n
<chr>,<int>
CDH1,665
VIM,596
BAX,511
VEGFA,468
BCL2L1,464
FN1,463


# Adductome IntegrationXXX
QIAGEN IPA Interpret predicts which upstream regulators are activated or inhibited to explain the up-regulated and down-regulated molecules observed in the dataset. Knowledge of these regulatory mechanisms can help you understand the biological activities occurring in the tissues or cells that are being studied.

 

(For an overview of IPA, see this article; for an overview of IPA Interpret see this article).

 
The term “upstream regulator” refers to any molecule that can affect the expression, transcription, or phosphorylation of another molecule. Upstream regulators encompass the gamut of molecule types found in the literature, from transcription factors to cytokines, microRNAs, receptors, kinases, chemicals, and drugs. Note that phosphorylation information is only used when analyzing phosphoproteomics data.

In [4]:
# getting sig genes
sig_genes_df = results_df %>%
    # filtering for only sig genes
    filter(P.Adj < 0.05 & log2FC > log2(1.5) | P.Adj < 0.05 & log2FC < -log2(1.5)) %>%
    dplyr::rename(ENSEMBL_ID = Gene)

# adding in gene names
sig_genes_df$Gene_Name = mapIds(org.Hs.eg.db, keys = sig_genes_df$ENSEMBL_ID, keytype = "ENSEMBL", column = "SYMBOL") 

# seeing if those gene names correpond to those with adducts post thc exposure
final_sig_df = sig_genes_df %>%
    mutate(Adduct = ifelse(Gene_Name %in% thc_adducts_df$Gene_Name, "Yes", "No"),
          # determining if any of those genes had upstream reugulators
          Regulator = ifelse(Gene_Name %in% cleaned_reg_df$Target_Molecules, "Yes", "No")) 
    

final_sig_df = final_sig_df[,c(1,6,5,7,8,2:4)]

head(final_sig_df)

'select()' returned 1:many mapping between keys and columns



,ENSEMBL_ID,Gene_Name,Comparison,Adduct,Regulator,log2FC,P.Value,P.Adj
,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>
1,ENSG00000161055,SCGB3A1,D8THCQ vs. Control,No,Yes,5.187423,6.699531e-22,1.406165e-17
2,ENSG00000149021,SCGB1A1,D8THCQ vs. Control,No,Yes,4.153279,2.655023e-18,2.786314e-14
3,ENSG00000124237,CIMIP1,D8THCQ vs. Control,No,No,3.514441,3.981380e-16,1.671304e-12
4,ENSG00000160472,TMEM190,D8THCQ vs. Control,No,No,3.658307,3.763892e-16,1.671304e-12
5,ENSG00000164972,SPMIP6,D8THCQ vs. Control,No,Yes,3.759643,2.399373e-16,1.671304e-12
6,ENSG00000105519,CAPS,D8THCQ vs. Control,No,Yes,3.127536,2.038872e-14,7.132315e-11


In [5]:
final_sig_df %>%
    filter(Adduct == 'Yes') %>%
    dplyr::select(Gene_Name) %>%
    unique() %>%
    count()

final_sig_df %>%
    filter(Regulator == 'Yes') %>%
    dplyr::select(Gene_Name) %>%
    unique() %>%
    count()

n
<int>
23


n
<int>
3995


Based on this information, there were 23 genes that formed DNA adducts after 24hrs of exposure to THCQ.

In [6]:
genes_of_interest = final_sig_df %>%
    filter(Adduct == 'Yes' & Regulator == 'Yes') %>%
    dplyr::select(Gene_Name) %>%
    unique() %>% 
    pull() %>%
    sort() 

length(genes_of_interest)
genes_of_interest

[1] 20

[1] "CCDC170"  "EXD3"     "FILIP1L"  "FYCO1"    "HSP90AA1" "INTS1"   
 [7] "ITPR3"    "LIN54"    "NCAPD3"   "PKN1"     "PSME1"    "SETX"    
[13] "TANC2"    "TTI1"     "UBR4"     "USP9X"    "WDR17"    "ZBTB21"  
[19] "ZFYVE26"  "ZNF581"

Based on this information, there were 20 genes that formed DNA adducts and had upstream regulators after 24hrs of exposure to THCQ. Now I need to figure out who those regulators are.

In [7]:
dna_adduct_regulated_df = upstream_reg_df[,c(1,3,4,7)] %>%
    separate_rows(Target_Molecules, sep = ",") %>%
    filter(Target_Molecules %in% genes_of_interest) %>%
    dplyr::rename(Target_Genes = Target_Molecules) %>%
    arrange(Upstream_Regulator)

head(dna_adduct_regulated_df)

Upstream_Regulator,Molecule_Type,Predicted_Activation_State,Target_Genes
<chr>,<chr>,<chr>,<chr>
"1,2-dithiole-3-thione",chemical reagent,Activated,HSP90AA1
10:0/10:0/10:0 triacylglycerol,chemical - endogenous mammalian,Inhibited,HSP90AA1
5-fluorouracil,chemical drug,Inhibited,PSME1
5-hydroxytryptamine,chemical - endogenous mammalian,NA,PKN1
8-bromo-cAMP,chemical reagent,NA,ITPR3
ACHE,enzyme,NA,USP9X


In [8]:
# exporting
write.xlsx(dna_adduct_regulated_df, paste0(Output,"/", "Regulated_Adduct_Genes_", cur_date, ".xlsx"), 
           rowNames = FALSE)